In [44]:
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import os
import json
import requests

In [45]:
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini_base_url = os.getenv("GEMINI_BASE_URL")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
if gemini_base_url:
    print(f"Gemini base url exists and ends with {gemini_base_url[-7:]}")

gemini = OpenAI(api_key=google_api_key, base_url=gemini_base_url)

Google API Key exists and begins AQ
Gemini base url exists and ends with openai/


In [54]:
system_check_message_prompt = """
You are an expert customer service sentiment analyst. Your task is to evaluate the tone and emotion of incoming client messages.

Analyze the client message provided below and return your assessment strictly in the following JSON format without any extra text or markdown formatting outside the JSON:

{
  "score": [An integer from 1 to 10, where 1 is extremely calm/happy and 10 is extremely angry/furious],
  "category": ["cool" if the score is between 1 and 6, or "angry" if the score is between 7 and 10],
  "reasoning": "A brief, one-sentence explanation for the assigned score based on word choice, punctuation, or context."
}
"""

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

def review_client_message(message, history): 
    messages = [{"role": "system", "content": system_check_message_prompt}] + history + [{"role": "user", "content": message}]
    
    response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)

    response_dict = json.loads(response.choices[0].message.content)
    response_score = response_dict["score"]
    
    if response_score > 6:
        payload = {"user": pushover_user, "token": pushover_token, "message": f"""Your client is angry ! {message}"""}
        requests.post(pushover_url, data=payload)
        
        score_messages = history + [{"role": "user", "content": f"The client wrote: '{message}'. Roleplay as a helpful customer support agent. Acknowledge their issue politely and assure them we are actively investigating/resolving it. Make it as a message reply not a email or something. Respond directly to the user in the first person. Do not wrap responses in quotation marks."}]
        
        response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=score_messages)
        content = response.choices[0].message.content
        if content.startswith('"') and content.endswith('"'):
            content = content[1:-1].strip()

            return content
        else:
            return content
    else:
         score_messages = history + [{"role": "user", "content": f"The client wrote: '{message}'. Roleplay as a helpful customer support agent. Reply warmly, positively, and helpfully to their message. Respond directly to the user in the first person. Do not wrap responses in quotation marks."}]
         
         response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=score_messages)
         content = response.choices[0].message.content
         
         if content.startswith('"') and content.endswith('"'):
            content = content[1:-1].strip()

            return content
         else:
            return content
        

Pushover token found and looks good


In [55]:
gr.ChatInterface(review_client_message).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
